In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import infercnvpy as cnv
import matplotlib.pyplot as plt
import gc


sc.settings.set_figure_params(figsize=(5, 5))
sc.logging.print_header()

In [ ]:
# data dir
input_dir = " "
output_dir = " "

cell_type_color = {"Epithelia":"#A6CEE3","Fibroblast":"#FDBF6F","Endothelia":"#B2DF8A","Acinar_cell":"#984EA3","SMC&Pericyte":"#33A02C","T&NK_cell":"#1F78B4","B_cell":"#FF7F00",
                   "Plasma_cell":"#FB9A99","Myeloid_cell":"#CAB2D6","Mast_cell":"#6A3D9A","Neutrophils":"#FFFF99","Hepatocyte":"#9ec9e1","Melanoma_cell":"#E66F00","Neuron":"#1d92c0",
                   "Glial_cell":"#B15928","Osteoblastic_cell":"#fcc5c1","Alpha_cell":"#42aa5e","Bela_cell":"#c22b86","undefine":"#A6CEE3"}

os.listdir(input_dir)

In [ ]:
# pre work
tumor_code = "BRCA"
cluster_method = "leiden"
bbknn_ridge = True
batch_method = "bbknn"
regress = False
resolution = 5
random_state = 123


In [ ]:
print(f"############################# {tumor_code} process ##########################################")

# data read
scRNA_infercnv = sc.read_h5ad(f"{input_dir}/{tumor_code}/scRNA_infercnv.h5ad")

# dir create
output_file = f"{output_dir}/{tumor_code}"
os.makedirs(output_file, exist_ok=True)

In [ ]:
# data plot
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw=dict(wspace=0.5))
cnv.pl.umap(scRNA_infercnv,color="cnv_leiden",legend_loc="on data",legend_fontoutline=2,show=False,ax=ax1)
cnv.pl.umap(scRNA_infercnv, color="cnv_score",show=False,ax=ax2)
cnv.pl.umap(scRNA_infercnv, color="cell_type",legend_loc="on data",show=False,ax=ax3)

In [ ]:
# tumor ano
scRNA_infercnv_predict = scRNA_infercnv.copy()
scRNA_infercnv_predict.obs["cnv_status"] = "normal"
## TGCT
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["14"]), "cnv_status"] = "tumor"
## BRCA
scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["21","23","13","22","16","12","18"]), "cnv_status"] = "tumor"
## PAAD
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["11","9","22","36","38","37","23","29","20","38","26","32","39","15","25","19","27","31"]), "cnv_status"] = "tumor"
## THCA
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["5","21","20","25","8","9","23","27","18"]), "cnv_status"] = "tumor"
## CRC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["23","3","24","27","25","20"]), "cnv_status"] = "tumor"
## ANS
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["22", "23", "13"]), "cnv_status"] = "tumor"
## HNSC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["26", "27", "20","17","29","25","35","31","18","24","15","32","30","28","33","22","23","34"]), "cnv_status"] = "tumor"
## ESCC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["11","16","24","29","17","27","25","20"]), "cnv_status"] = "tumor"
## KIRC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["10","17","16","25","26","22"]), "cnv_status"] = "tumor"
## NPC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["0","1","9"]), "cnv_status"] = "tumor"
## SARC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["1","12","16","4","5"]), "cnv_status"] = "tumor"
## ACC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["0","1","3","8","11"]), "cnv_status"] = "tumor"
## LUAD
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["22","34","28","31","23","39","30","36","29","27","15","37","25","33","38","6"]), "cnv_status"] = "tumor"
## GC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["23","24","18","19","14","22"]), "cnv_status"] = "tumor"
## MA
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["23","11","5","6","4","18","13","19","25","7","22","26","8","12","21"]), "cnv_status"] = "tumor"
## PNET
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["21","14","27","23","19","30","22","29","24","4","17","18","31"]), "cnv_status"] = "tumor"
## CESC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["18","16","21","6","17"]), "cnv_status"] = "tumor"
## LSCC
# scRNA_infercnv_predict.obs.loc[scRNA_infercnv_predict.obs["cnv_leiden"].isin(["2","3","6","15","22","7","26","12","13","1","17"]), "cnv_status"] = "tumor"

os.chdir(output_file)
sc.pl.umap(scRNA_infercnv_predict, color="cnv_status",show = False,palette = {"tumor":"#E41A1C","normal":"#0072B5"},legend_loc='none')
plt.savefig(f'{output_file}/scRNA_umap_cnv_status.png', dpi=3000)
sc.pl.umap(scRNA_infercnv_predict, color='cnv_score',show = False,legend_loc='none')
plt.savefig(f'{output_file}/scRNA_umap_cnv_score.png', dpi=3000)
sc.pl.umap(scRNA_infercnv_predict, color='cell_type',show = False,legend_loc='none',palette=cell_type_color)
plt.savefig(f'{output_file}/scRNA_umap_cell_type.png', dpi=3000)

In [ ]:
# data save
scRNA_infercnv_meta = pd.DataFrame(scRNA_infercnv_predict.obs)
scRNA_infercnv_meta.to_csv(f"{output_file}/scRNA_infercnv_meta.csv", index=False)
scRNA_infercnv_predict.write_h5ad(f"{output_file}/scRNA_infercnv_predict.h5ad", compression="gzip")